In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from collections import Counter
import random
import re
import datasets
import tqdm
import math
from functools import partial
import math
import argparse
import os
import collections
import json
import sentencepiece
import shutil
import copy
import multiprocessing
import transformers
from dataclasses import dataclass, field
from evaluate import load

# set "high" if you have a GPU with compute capability >= 8.0 else "highest"
torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

/home/hoon/miniconda3/envs/cse402/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Training config

In [2]:
## you can modify some options such as batch_size, depending on your environments  

training_config = {
    "batch_size": 16,
    "epochs": 3,
    "lr": 1e-3, #! 1e-4 -> 1e-3
    "warmup_steps": 50,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "gradient_accumulate_steps": 1,
}

# Dataset load

In [3]:
dataset = datasets.load_dataset("lemon-mint/korean_english_parallel_wiki_augmented_v1",split="train")
dataset = dataset.filter(lambda x: len(x['english']) < 8192 and len(x['english']) > 128 and len(x['korean']) < 8192 and len(x['korean']) > 128)
valid_set = dataset.select(range(10000))
train_set = dataset.select(range(10000, 110000))

tokenizer = transformers.AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ko-en")
additional_special_tokens = {}
if tokenizer.pad_token is None:
    additional_special_tokens["pad_token"] = "<pad>"
if tokenizer.eos_token is None:
    additional_special_tokens["eos_token"] = "</s>"
if tokenizer.bos_token is None:
    additional_special_tokens["bos_token"] = "<s>"
tokenizer.add_special_tokens(additional_special_tokens)

def collate_fn(batch):
    english_corpus = [item["english"] for item in batch]
    korean_corpus = [item["korean"] for item in batch]
    english_corpus = tokenizer(english_corpus, padding=True, truncation=True, return_tensors="pt", max_length=512, pad_to_multiple_of=64)
    # korean_corpus = tokenizer(korean_corpus, padding=True, truncation=True, return_tensors="pt", max_length=512, pad_to_multiple_of=64)
    korean_corpus = tokenizer(korean_corpus, padding=True, truncation=True, return_tensors="pt", max_length=511)
    

    # labels = korean_corpus["input_ids"].clone()
    # labels[korean_corpus['attention_mask'].eq(0)] = -100

    batch_size, target_len = korean_corpus["input_ids"].shape
    decoder_input_ids = torch.full(
        (batch_size, target_len + 1),
        tokenizer.pad_token_id,
        dtype=korean_corpus["input_ids"].dtype,
    )
    decoder_input_ids[:, 0] = tokenizer.bos_token_id
    decoder_input_ids[:, 1:] = korean_corpus["input_ids"]

    labels = decoder_input_ids.clone()
    label_attention_mask = torch.zeros_like(decoder_input_ids)
    label_attention_mask[:, 0] = 1
    label_attention_mask[:, 1:] = korean_corpus["attention_mask"]
    labels[label_attention_mask.eq(0)] = -100


    return {
        "encoder_input_ids": english_corpus["input_ids"],
        "encoder_attention_mask": english_corpus["attention_mask"],
        # "decoder_input_ids": korean_corpus["input_ids"],
        "decoder_input_ids": decoder_input_ids,
        # "labels": korean_corpus["input_ids"],
        "labels": labels
    }



/home/hoon/miniconda3/envs/cse402/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


### Model implement

In [4]:
@dataclass
class ModelConfig(object):
    vocab_size: int = field(default=50000)
    encoder_hidden_dim: int = field(default=512) # hidden dimention of encoder lstm
    decoder_hidden_dim: int = field(default=512) # hidden dimention of decoder lstm
    hidden_dim: int = field(default=512) # hidden dimention of other module like attention
    embed_dim: int = field(default=512) # embedding dimention
    pad_idx: int = field(default=0)
    sos_idx: int = field(default=1)
    eos_idx: int = field(default=2)
    n_layers: int = field(default=1)
    dropout: float = field(default=0.1)

    attention_type:str = field(default="global")
    window_size: int = field(default=10)
    sigma_ratio: float = field(default=2.0)

    do_input_feeding: bool = field(default=True)

class GlobalAttention(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        self.query_proj = nn.Linear(config.decoder_hidden_dim, config.hidden_dim, bias=False)
        self.key_proj = nn.Linear(config.encoder_hidden_dim * 2, config.hidden_dim, bias=False)
        self.value_proj = nn.Linear(config.encoder_hidden_dim * 2, config.hidden_dim, bias=False)
        self.output_proj = nn.Linear(config.hidden_dim, config.decoder_hidden_dim, bias=False)
        
        self.dropout = nn.Dropout(config.dropout)
        self.scale = np.sqrt(config.hidden_dim)

    def forward(self, decoder_hidden_query, encoder_outputs, encoder_attention_mask):
        query = self.query_proj(decoder_hidden_query)
        key = self.key_proj(encoder_outputs)
        value = self.value_proj(encoder_outputs)
        
        # fill here for global attention forward
        # shape hint:
        # context: (batch, 1, hidden_dim)
        ######
        
        ## YOUR CODES 

        #!=======================================================================================
        # calculate scaled dot-product attention scores
        attn_scores = torch.bmm(query, key.transpose(1, 2)) / self.scale
        # mask padding tokens
        attn_scores = attn_scores.masked_fill(
            encoder_attention_mask.unsqueeze(1).eq(0),
            torch.finfo(attn_scores.dtype).min,
        )
        # softmax to get weights, then compute context vector
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context = torch.bmm(attn_weights, value)
        #!=======================================================================================
        
        ######
        output_context = self.output_proj(context)

        return output_context

class LocalAttention(GlobalAttention):
    def __init__(self, config: ModelConfig):
        super().__init__(config) 
        self.window_size = config.window_size
        self.location_proj_up = nn.Linear(config.decoder_hidden_dim, config.hidden_dim, bias=False)
        self.location_proj_down = nn.Linear(config.hidden_dim, 1, bias=False)
        self.sigma = self.window_size / config.sigma_ratio

    def forward(self, decoder_hidden_query, encoder_outputs, encoder_attention_mask):
        key, value, attn_mask, gaussian_penalty = self._gather_local_context(decoder_hidden_query, encoder_outputs, encoder_attention_mask)
        query = self.query_proj(decoder_hidden_query)
        key = self.key_proj(key)
        value = self.value_proj(value)

        # fill here for local attention forward
        # shape hint:
        # context: (batch, 1, hidden_dim)
        ######
        
        ## YOUR CODES 

        #!=======================================================================================
        # scaled dot-product scores with gaussian penalty (applied in log space)
        attn_scores = torch.bmm(query, key.transpose(1, 2)) / self.scale
        attn_scores = attn_scores + gaussian_penalty.unsqueeze(1)
        # mask out of window and padding positions
        attn_scores = attn_scores.masked_fill(
            attn_mask.unsqueeze(1).eq(0),
            torch.finfo(attn_scores.dtype).min,
        )
        # softmax to get weights, then compute context vector
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context = torch.bmm(attn_weights, value)
        #!=======================================================================================
        
        ######
        output_context = self.output_proj(context)

        return output_context

    def _gather_local_context(self, decoder_hidden_query, encoder_outputs, encoder_attention_mask):
        device = encoder_outputs.device
        src_len = encoder_attention_mask.sum(dim=-1).unsqueeze(-1)

        # fill here for local context window
        # shape hint:
        # local_key: (batch, window_size * 2 + 1, hidden_dim)
        # local_value: (batch, window_size * 2 + 1, hidden_dim)
        # local_attn_mask: (batch, window_size * 2 + 1)
        # gaussian_penalty: (batch, window_size * 2 + 1)
        ######
        
        ## YOUR CODES 

        #!=======================================================================================
        batch_size, src_seq_len, hidden_dim = encoder_outputs.shape
        dtype = encoder_outputs.dtype

        # get actual source lengths per batch
        valid_len = src_len.squeeze(-1).long().to(device)
        valid_len = valid_len.clamp(min=1)
        valid_len_float = valid_len.to(dtype=dtype)

        # predict center position (pt) of the local window
        location_hidden = torch.tanh(self.location_proj_up(decoder_hidden_query))
        position_ratio = torch.sigmoid(self.location_proj_down(location_hidden))

        # reshape to [B]
        position_ratio = position_ratio.reshape(batch_size, -1)[:, 0]

        # center index in valid range
        predicted_position = (valid_len_float - 1) * position_ratio  # [B]

        # local window indices
        offsets = torch.arange(
            -self.window_size,
            self.window_size + 1,
            device=device,
            dtype=torch.long,
        )  # [2w + 1]

        center_position = predicted_position.round().long().unsqueeze(-1)  # [B, 1]
        local_positions = center_position + offsets.unsqueeze(0)           # [B, 2w + 1]

        # mask to validate source sequence boundaries
        local_attn_mask = (
            (local_positions >= 0)
            & (local_positions < valid_len.unsqueeze(-1))
        )  # [B, 2w + 1]

        # gather key and value vectors within the window
        gather_positions = local_positions.clamp(0, src_seq_len - 1)

        gather_index = gather_positions.unsqueeze(-1).expand(
            -1, -1, hidden_dim
        )  # [B, 2w + 1, H]

        local_key = encoder_outputs.gather(dim=1, index=gather_index)
        local_value = local_key

        # compute gaussian penalty (added in log space for numerical stability)
        sigma = max(float(self.sigma), 1e-6)
        gaussian_penalty = -(
            (local_positions.to(dtype=dtype) - predicted_position.unsqueeze(-1)) ** 2
        ) / (2 * (sigma ** 2))
        #!=======================================================================================
        
        ######

        return local_key, local_value, local_attn_mask, gaussian_penalty

class Encoder(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        
        self.encoder = nn.LSTM(
            input_size=config.embed_dim,
            hidden_size=config.encoder_hidden_dim,
            num_layers=config.n_layers,
            dropout=config.dropout if config.n_layers > 1 else 0,
            bidirectional=True,
            batch_first=True
        )

        self.h_dec_proj = nn.Linear(config.encoder_hidden_dim * 2, config.decoder_hidden_dim)
        self.c_dec_proj = nn.Linear(config.encoder_hidden_dim * 2, config.decoder_hidden_dim)

    def forward(self, input_embeds, attention_mask):

        # Fill here for encoder forward
        # shape hint
        # input_embeds: (batch, src_seq_len, embed_dim)
        # attention_mask: (batch, src_seq_len)
        # encoder_output: (batch, src_seq_len, hidden_dim)
        # h_enc: (n_layers, batch, decoder_hidden_dim)
        # c_enc: (n_layers, batch, decoder_hidden_dim)
        # hint for implementation
        # 1. use nn.utils.rnn.pack_padded_sequence to packing inputs for rnn series, see https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html
        #    failure to properly handle padding will result in a penalty.
        # 2. lstm cell state and hidden state will be doubled because of bidirectional lstm.
        #    decoder will be unidirectional for causal language modeling. 
        #    handle the hidden state and cell state to be same as decoder.
        ######
        
        ## YOUR CODES 

        #!=======================================================================================
        # pack padded sequence to ignore padding tokens in LSTM
        lengths = attention_mask.sum(dim=-1).to(dtype=torch.long).cpu()
        packed_embeds = nn.utils.rnn.pack_padded_sequence(
            input_embeds,
            lengths,
            batch_first=True,
            enforce_sorted=False,
        )
        packed_output, (h_enc_bidir, c_enc_bidir) = self.encoder(packed_embeds)
        encoder_output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=input_embeds.size(1),
        )

        batch_size = input_embeds.size(0)
        h_enc_bidir = h_enc_bidir.view(
            self.config.n_layers, 2, batch_size, self.config.encoder_hidden_dim
        )
        c_enc_bidir = c_enc_bidir.view(
            self.config.n_layers, 2, batch_size, self.config.encoder_hidden_dim
        )

        h_cat = torch.cat([h_enc_bidir[:, 0], h_enc_bidir[:, 1]], dim=-1)
        c_cat = torch.cat([c_enc_bidir[:, 0], c_enc_bidir[:, 1]], dim=-1)

        h_enc = self.h_dec_proj(h_cat)
        c_enc = self.c_dec_proj(c_cat)
        #!=======================================================================================
        
        ######

        return encoder_output, (h_enc, c_enc)

class Decoder(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        self.decoder = nn.LSTM(
            input_size=config.embed_dim + config.hidden_dim if config.do_input_feeding else config.embed_dim,
            hidden_size=config.decoder_hidden_dim,
            num_layers=config.n_layers,
            dropout=config.dropout if config.n_layers > 1 else 0,
            batch_first=True
        )
        match config.attention_type:
            case "local":
                self.attention = LocalAttention(config)
            case "global":
                self.attention = GlobalAttention(config)
            case _:
                raise ValueError(f"Unknown attention type: {config.attention_type}")
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_embeds, encoder_outputs, h_enc, c_enc, attention_mask):
        decoder_output, (h_dec, c_dec) = self.decoder(input_embeds, (h_enc, c_enc))
        attention_context = self.attention(decoder_output, encoder_outputs, attention_mask)
        decoder_output = decoder_output + attention_context

        return decoder_output, attention_context, (h_dec, c_dec)

class Seq2Seq(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        self.embedding = nn.Embedding(config.vocab_size, config.embed_dim, padding_idx=config.pad_idx)
        
        self.encoder = Encoder(config)
        self.decoder = Decoder(config)
        
        self.lm_head = nn.Linear(config.hidden_dim, config.vocab_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, encoder_input_ids, encoder_attention_mask, decoder_input_ids, labels=None, cache=None):
        if cache is None:
            encoder_input_embeds = self.embedding(encoder_input_ids)
            encoder_outputs, (h_enc, c_enc) = self.encoder(encoder_input_embeds, encoder_attention_mask)

            current_h_dec, current_c_dec = h_enc, c_enc
            prev_attn_context = None
        else:
            encoder_outputs, current_h_dec, current_c_dec, prev_attn_context = cache

        batch_size, tgt_len = decoder_input_ids.shape
        decoder_input_embeds = self.embedding(decoder_input_ids)

        if prev_attn_context is None:
            prev_attn_context = torch.zeros((batch_size, 1, self.config.decoder_hidden_dim)).to(decoder_input_embeds)
        
        outputs = [] 

        for t in range(tgt_len):
            # fill here for decoder forward
            ######
        
            ## YOUR CODES 

            #!=======================================================================================
            current_input_embeds = decoder_input_embeds[:, t:t + 1]
            # concat previous context vector if input feeding is enabled
            if self.config.do_input_feeding:
                current_input_embeds = torch.cat([current_input_embeds, prev_attn_context], dim=-1)
            # step decoder and update hidden state and context vector
            decoder_output, prev_attn_context, (current_h_dec, current_c_dec) = self.decoder(
                current_input_embeds,
                encoder_outputs,
                current_h_dec,
                current_c_dec,
                encoder_attention_mask,
            )
            #!=======================================================================================
            
            ######
            outputs.append(decoder_output)
            

        outputs = torch.cat(outputs, dim=1)
        
        lm_logits = self.lm_head(outputs)
    
        loss = None
        if labels is not None:
            # for cross entropy loss
            # loss must be scalar
           
            labels_for_loss = labels[:, 1:].contiguous()
            lm_logits_for_loss = lm_logits[:, :-1, :].contiguous()
            loss = F.cross_entropy(lm_logits_for_loss.view(-1, self.config.vocab_size), labels_for_loss.view(-1))
           
            return loss
        else:
            return lm_logits, (encoder_outputs, current_h_dec, current_c_dec, prev_attn_context)

    @torch.no_grad()
    def generate(
        self,
        encoder_input_ids: torch.LongTensor,
        encoder_attention_mask: torch.LongTensor,
        max_new_tokens: int = 256,
    ):
        batch_size, _ = encoder_input_ids.shape
        device = encoder_input_ids.device
        eos = self.config.eos_idx

        unfinish_flag = torch.ones(batch_size, dtype=torch.long, device=device)
        cache = None
        decoder_input_ids = torch.full((batch_size, 1), self.config.sos_idx, dtype=torch.long, device=device)

        for _ in range(max_new_tokens):
            # fill here for causal generation
           ######
        
            ## YOUR CODES 
            #!=======================================================================================
            # incremental decoder step using previous prediction
            lm_logits, cache = self.forward(
                encoder_input_ids=encoder_input_ids,
                encoder_attention_mask=encoder_attention_mask,
                decoder_input_ids=decoder_input_ids[:, -1:],
                cache=cache,
            )
            # greedy search step
            next_tokens = lm_logits[:, -1, :].argmax(dim=-1)
            next_tokens = next_tokens * unfinish_flag + self.config.pad_idx * (1 - unfinish_flag)
            decoder_input_ids = torch.cat([decoder_input_ids, next_tokens.unsqueeze(-1)], dim=-1)
            # update finished flag and check for EOS
            unfinish_flag = unfinish_flag * next_tokens.ne(eos).long()
            if unfinish_flag.max() == 0:
                break
            #!=======================================================================================
            
            ######
        return decoder_input_ids


In [5]:
def train(model, train_dataset, valid_dataset, collate_fn, train_args, prefix):
    optimizer = optim.Adam(model.parameters(), lr=train_args["lr"])

    train_dataloader = DataLoader(train_dataset, batch_size=train_args['batch_size'], shuffle=True, collate_fn=collate_fn, num_workers=os.cpu_count())
    valid_dataloader = DataLoader(valid_dataset, batch_size=train_args['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=os.cpu_count())

    total_steps = len(train_dataloader) * train_args['epochs']

    num_training_steps = train_args['epochs'] * (len(train_dataloader) // train_args['gradient_accumulate_steps'])
    scheduler = transformers.get_scheduler(
        name="cosine",
        optimizer=optimizer,
        num_warmup_steps=train_args['warmup_steps'],
        num_training_steps=num_training_steps
    )

    best_loss = 987654321
    optimizer.zero_grad()

    output_path = os.path.join("output", prefix)
    os.makedirs(output_path, exist_ok=True)
    with open(os.path.join(output_path, "train_args.json"), "w") as f:
        json.dump(train_args, f)

    pbar = tqdm.tqdm(total=total_steps, desc="training")
    for epoch in range(train_args['epochs']):
        pbar.set_description(f"Epoch {epoch+1}/{train_args['epochs']}")
        move_avg_loss = []
        model.train()
        for i, batch in enumerate(train_dataloader):
            batch = {k:v.to(train_args['device']) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}

            loss = model(**batch)
            loss = loss / train_args['gradient_accumulate_steps']
            if loss.size() != torch.Size([]):
                loss = loss.mean()
            loss.backward()
            
            if (i+1) % train_args['gradient_accumulate_steps'] == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()

            move_avg_loss.append(loss.item()) 
            if len(move_avg_loss) > 100: move_avg_loss.pop(0)
            pbar.set_postfix_str(f"loss: {sum(move_avg_loss)/len(move_avg_loss):.04f} lr: {optimizer.param_groups[0]['lr']:.2e}")
            pbar.update(1)
        
        model.eval()
        with torch.no_grad():
            eval_loss = 0
            for i, batch in enumerate(valid_dataloader):
                batch = {k:v.to(train_args['device']) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}
                loss_val = model(**batch)
                if loss_val.size() != torch.Size([]):
                    loss_val = loss_val.mean()
                eval_loss += loss_val.item()
                pbar.set_postfix_str(f"val_loss: {eval_loss / (i+1):.04f}")
        eval_loss /= len(valid_dataloader)
        pbar.write(f"Validation Loss: {eval_loss:.04f}")

        if eval_loss < best_loss:
            best_loss = eval_loss
            
            torch.save(model.state_dict(), os.path.join(output_path,"best_model.pth"))
            pbar.write(f"Model Saved best loss: {best_loss:.04f}")

    pbar.close()

def evaluate(model, dataset, tokenizer, collate_fn, train_args):
    model.eval()
    dataloader = DataLoader(dataset, batch_size=train_args['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=os.cpu_count())
    
    answers = []
    predicts = []
    for i, batch in enumerate(tqdm.tqdm(dataloader, desc="Evaluating")):
        batch = {k:v.to(train_args['device']) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}
        gen_output = model.generate(
            encoder_input_ids=batch["encoder_input_ids"],
            encoder_attention_mask=batch["encoder_attention_mask"],
            max_new_tokens=512
        )
        pred = tokenizer.batch_decode(gen_output, skip_special_tokens=True)
        # ans = tokenizer.batch_decode(batch["labels"], skip_special_tokens=True)
        answer_ids = batch["labels"].masked_fill(batch["labels"].eq(-100), tokenizer.pad_token_id)
        ans = tokenizer.batch_decode(answer_ids, skip_special_tokens=True)

        if i == 0: # debugging
            print("pred1: ", pred[0:1])
            print("ans1:  ", ans[0:1])

            print("pred2: ", pred[1:2])
            print("ans2:  ", ans[1:2])
        
        answers.extend(ans)
        predicts.extend(pred)
    
    bleu = load("bleu")
    result = bleu.compute(predictions=predicts, references=answers)
    print(f"BLEU: {result['bleu']:.4f}")

In [6]:
config = ModelConfig(
    vocab_size=len(tokenizer),
    pad_idx=tokenizer.pad_token_id,
    sos_idx=tokenizer.bos_token_id,
    eos_idx=tokenizer.eos_token_id,
    n_layers=2,
    dropout=0.1,

    attention_type="global",
    do_input_feeding=False,
)

model = Seq2Seq(config).to(training_config["device"])
# model = model.to(torch.bfloat16)
# model.compile()
print(model)

train(
    model,
    train_set,
    valid_set,
    collate_fn,
    training_config,
    prefix="seq2seq_global_attention_no_input_feeding"
)

model.load_state_dict(torch.load(os.path.join("output", "seq2seq_global_attention_no_input_feeding", "best_model.pth")))
evaluate(
    model,
    valid_set,
    tokenizer,
    collate_fn,
    training_config
)

del model
torch.cuda.empty_cache()

Seq2Seq(
  (embedding): Embedding(65002, 512, padding_idx=65000)
  (encoder): Encoder(
    (encoder): LSTM(512, 512, num_layers=2, batch_first=True, dropout=0.1, bidirectional=True)
    (h_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
    (c_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
  )
  (decoder): Decoder(
    (decoder): LSTM(512, 512, num_layers=2, batch_first=True, dropout=0.1)
    (attention): GlobalAttention(
      (query_proj): Linear(in_features=512, out_features=512, bias=False)
      (key_proj): Linear(in_features=1024, out_features=512, bias=False)
      (value_proj): Linear(in_features=1024, out_features=512, bias=False)
      (output_proj): Linear(in_features=512, out_features=512, bias=False)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (lm_head): Linear(in_features=512, out_features=65002, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


Epoch 1/3:  33%|███▎      | 6250/18750 [1:01:10<1:49:06,  1.91it/s, val_loss: 2.4724]       

Validation Loss: 2.4724


Epoch 2/3:  33%|███▎      | 6250/18750 [1:01:10<1:49:06,  1.91it/s, val_loss: 2.4724]

Model Saved best loss: 2.4724


...

Model Saved best loss: 1.8376


Evaluating:   0%|          | 1/625 [00:04<48:00,  4.62s/it]

pred1:  ['5059 알루미늄 합금은 주로 마그네슘과 함께 건조된 알루미늄-매그네슘 합금입니다. 열처리에 의해 강화되지 않아 재료의 강화 또는 냉기계적 작동으로 인해 더욱 강해졌습니다. 열처리가 호전되면 5059년 융기에 융기된 것으로 간주되지는 못이 가능하며, 대부분의 기계적 강도는 기계적 강도를 초래할 수 있습니다. 5059개의 합금은 1999년 코러스 알루미니엄에서 연구자들이 호스루눔의 연구자들에 의해 느슨하게 관련되어 있습니다.']
ans1:   ['5059 알루미늄 합금은 주로 마그네슘으로 합금된 알루미늄-마그네슘 합금입니다. 열처리로 강화되지 않고, 대신 재료의 변형 경화 또는 냉간 기계 가공으로 강해집니다. 열처리가 강도에 큰 영향을 미치지 않기 때문에 5059는 용접이 용이하고 기계적 강도를 대부분 유지할 수 있습니다. 5059 합금은 1999년 코러스 알루미늄의 연구원들에 의해 밀접하게 관련된 5083 알루미늄 합금에서 유래했습니다.']
pred2:  ['쏜베리 동물 보호구역은 영국 사우스 요크셔 로더햄에 위치한 중간 크기의 동물 구조 및 복지 자선 단체입니다. 동물 보호 구역이자 쉼터이며, 목재와 농장 동물을 위한 임시 거처와 영구 돌봄을 제공하며, 비파괴적인 방지 정책을 따릅니다. 손베리는 1988년 스티브 배암포드가 설립했으며, 이 땅을 보호구역에 매각하고 동물의 일을 돌보는 것을 깨닫습니다. 샘베리는 보호구역에 있는 동안 5년 동안 소규모 카라반에 살았으며, 보호구역은 보호구역이 설립되었습니다. 노스 안스턴의 주요 성소는 운하, 난방, 마구간, 헛간이 조성되었습니다. 노스 안스턴의 주요 성소는 처음에는 케넬스, 배터리, 스테이블, 헛간을 포함하여 목재로 둘러싸인 구조물을 시작했습니다. 마구간은 나중에 워크솝의 버크스 농장으로 이전했지만, 이 시설은 다시 매각되었고, 그 시설은 다시 레이븐필드의 실버소프 팜에 있습니다.']
ans2:   ['쏜베리 동물 보호소는 영국 사우스 요크셔주 로더햄에 위치한 중간 규모의 동물 구조 및 복지 자

Evaluating: 100%|██████████| 625/625 [02:47<00:00,  3.73it/s]


BLEU: 0.1806


In [7]:
config = ModelConfig(
    vocab_size=len(tokenizer),
    pad_idx=tokenizer.pad_token_id,
    sos_idx=tokenizer.bos_token_id,
    eos_idx=tokenizer.eos_token_id,
    n_layers=2,
    dropout=0.1,

    attention_type="global",
)

model = Seq2Seq(config).to(training_config["device"])
# model = model.to(torch.bfloat16)
# model.compile()
print(model)

train(
    model,
    train_set,
    valid_set,
    collate_fn,
    training_config,
    prefix="seq2seq_global_attention"
)

model.load_state_dict(torch.load(os.path.join("output", "seq2seq_global_attention", "best_model.pth")))
evaluate(
    model,
    valid_set,
    tokenizer,
    collate_fn,
    training_config
)

del model
torch.cuda.empty_cache()

Seq2Seq(
  (embedding): Embedding(65002, 512, padding_idx=65000)
  (encoder): Encoder(
    (encoder): LSTM(512, 512, num_layers=2, batch_first=True, dropout=0.1, bidirectional=True)
    (h_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
    (c_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
  )
  (decoder): Decoder(
    (decoder): LSTM(1024, 512, num_layers=2, batch_first=True, dropout=0.1)
    (attention): GlobalAttention(
      (query_proj): Linear(in_features=512, out_features=512, bias=False)
      (key_proj): Linear(in_features=1024, out_features=512, bias=False)
      (value_proj): Linear(in_features=1024, out_features=512, bias=False)
      (output_proj): Linear(in_features=512, out_features=512, bias=False)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (lm_head): Linear(in_features=512, out_features=65002, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


Epoch 1/3:  33%|███▎      | 6250/18750 [1:03:13<2:06:41,  1.64it/s, val_loss: 2.3568]         

Validation Loss: 2.3568


Epoch 2/3:  33%|███▎      | 6250/18750 [1:03:14<2:06:41,  1.64it/s, val_loss: 2.3568]

Model Saved best loss: 2.3568


...

Model Saved best loss: 1.7863


Evaluating:   0%|          | 1/625 [00:05<58:01,  5.58s/it]

pred1:  ['5059 알루미늄 합금은 주로 마그네슘으로 간주됩니다. 열처리로 인해 강화되지 않고, 대신 빗물의 냉소 정비 또는 냉소한 기계적 작동으로 인해 강수량이 높아지는 경우가 많습니다. 열처리가 열성 대우가 발생하면 5059년은 기계적 강도 대부분을 쉽게 감지하고 유지하기 위해 나타났습니다. 5059 합금은 1999년 코러스 알루미니움에서 연구원들에 의해 균적으로 관련되어 있습니다.']
ans1:   ['5059 알루미늄 합금은 주로 마그네슘으로 합금된 알루미늄-마그네슘 합금입니다. 열처리로 강화되지 않고, 대신 재료의 변형 경화 또는 냉간 기계 가공으로 강해집니다. 열처리가 강도에 큰 영향을 미치지 않기 때문에 5059는 용접이 용이하고 기계적 강도를 대부분 유지할 수 있습니다. 5059 합금은 1999년 코러스 알루미늄의 연구원들에 의해 밀접하게 관련된 5083 알루미늄 합금에서 유래했습니다.']
pred2:  ['쏜베리 동물 보호 구역은 영국 사우스 요크셔주 로더햄에 위치한 중간 크기의 동물 구조 및 복지 자선입니다. 동물 보호 구역이자 쉼터로, 애도적인 피난처와 농장 동물을 위한 영구적인 치료를 제공하며, 비살상 정책을 증진하는 데 앞장섰습니다. 쏜베리는 1988년 스티브 배포드가 설립했으며, 이 땅을 사퇴하고 농장을 돌보는 땅을 사서 자신의 삶을 살았습니다. 팜포드는 보호구역이 보호구역에 살았습니다. 팜포드는 보호구역이 조성된 보호구역에 살았습니다. 노스 안스턴의 주요 보호구역 부지는 처음에는 케넬, 포털, 마구간, 헛간을 포함하여 신전을 시작했습니다. 마구간은 나중에 워크솝의 버크스 농장으로 이전했지만, 이 시설은 다시 매각되었고, 이 시설은 다시 매각되었고, 마구간은 현재 레이븐필드의 실버소프 팜에 있습니다.']
ans2:   ['쏜베리 동물 보호소는 영국 사우스 요크셔주 로더햄에 위치한 중간 규모의 동물 구조 및 복지 자선 단체입니다. 이곳은 동물 보호소이자 쉼터로, 애완 동물과 농장 동물에게 임시 보호와 영구적인 보살핌을 제공하며, 

Evaluating: 100%|██████████| 625/625 [02:50<00:00,  3.68it/s]


BLEU: 0.2015


In [ ]:
config = ModelConfig(
    vocab_size=len(tokenizer),
    pad_idx=tokenizer.pad_token_id,
    sos_idx=tokenizer.bos_token_id,
    eos_idx=tokenizer.eos_token_id,
    n_layers=2,
    dropout=0.1,

    attention_type="local",
)

model = Seq2Seq(config).to(training_config["device"])
# model = model.to(torch.bfloat16)
# model.compile()
print(model)

train(
    model,
    train_set,
    valid_set,
    collate_fn,
    training_config,
    prefix="seq2seq_local_attention"
)

model.load_state_dict(torch.load(os.path.join("output", "seq2seq_local_attention", "best_model.pth")))
evaluate(
    model,
    valid_set,
    tokenizer,
    collate_fn,
    training_config
)

del model
torch.cuda.empty_cache()

Seq2Seq(
  (embedding): Embedding(65002, 512, padding_idx=65000)
  (encoder): Encoder(
    (encoder): LSTM(512, 512, num_layers=2, batch_first=True, dropout=0.1, bidirectional=True)
    (h_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
    (c_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
  )
  (decoder): Decoder(
    (decoder): LSTM(1024, 512, num_layers=2, batch_first=True, dropout=0.1)
    (attention): LocalAttention(
      (query_proj): Linear(in_features=512, out_features=512, bias=False)
      (key_proj): Linear(in_features=1024, out_features=512, bias=False)
      (value_proj): Linear(in_features=1024, out_features=512, bias=False)
      (output_proj): Linear(in_features=512, out_features=512, bias=False)
      (dropout): Dropout(p=0.1, inplace=False)
      (location_proj_up): Linear(in_features=512, out_features=512, bias=False)
      (location_proj_down): Linear(in_features=512, out_features=1, bias=False)
    )
    (dropout): Dropout(p=0.1,

Epoch 1/3:  33%|███▎      | 6250/18750 [1:18:13<2:27:30,  1.41it/s, val_loss: 4.0861]         

Validation Loss: 4.0861


Epoch 2/3:  33%|███▎      | 6250/18750 [1:18:14<2:27:30,  1.41it/s, val_loss: 4.0861]

Model Saved best loss: 4.0861


...

Model Saved best loss: 3.7329


Evaluating:   0%|          | 1/625 [00:06<1:07:02,  6.45s/it]

pred1:  ['2개의 섬유 사이클은 섬유 섬유로, 섬유 섬유로 교체된 섬유로 둘러싸여 있습니다. 이 복합체는 목재로 덮인 목재로 둘러싸여 있으며, 목재 섬유로 둘러싸여 있습니다. 이 복합체는 목재로 덮인 목재로 둘러싸여 있으며, 목재 섬유로 둘러싸여 있습니다. 이 복합체는 목재로 덮인 목재로 둘러싸여 있으며, 목재 섬유로 둘러싸여 있습니다. 이 복합체는 목재로 덮인 목재로 둘러싸여 있으며, 목재 섬유로 둘러싸여 있습니다.']
ans1:   ['5059 알루미늄 합금은 주로 마그네슘으로 합금된 알루미늄-마그네슘 합금입니다. 열처리로 강화되지 않고, 대신 재료의 변형 경화 또는 냉간 기계 가공으로 강해집니다. 열처리가 강도에 큰 영향을 미치지 않기 때문에 5059는 용접이 용이하고 기계적 강도를 대부분 유지할 수 있습니다. 5059 합금은 1999년 코러스 알루미늄의 연구원들에 의해 밀접하게 관련된 5083 알루미늄 합금에서 유래했습니다.']
pred2:  ['프랜시스 라이트는 잉글랜드 케이프타운의 외딴 지역에 위치한 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 야외 하우스입니다. 이 저택은 케이프타운의 외딴 지역인 우드 파크에서 유래한 것으로 알려져 있습니다. 이 저택은 케이프타운의 외딴 지역인 우드 하우스에서 유래했으며, 방문객들에게 피난처를 제공했습니다. 이 저택은 인근 플랫 파크에서 태어났습니다. 그의 아버지인 프랜시스 스튜어트는 잉글랜드의 외딴 지역인 캐슬 파크에서 태어났습니다.']
ans2:   ['쏜베리 동물 보호소는 영국 사우스 요크셔주 로더햄에 위치한 중간 규모의 동물 구조 및 복지 자선 단체입니다. 이곳은 동물 보호소이자 쉼터로, 애완 동물과 

Evaluating: 100%|██████████| 625/625 [04:48<00:00,  2.17it/s]


BLEU: 0.0129


In [ ]:
config = ModelConfig(
    vocab_size=len(tokenizer),
    pad_idx=tokenizer.pad_token_id,
    sos_idx=tokenizer.bos_token_id,
    eos_idx=tokenizer.eos_token_id,
    n_layers=2,
    dropout=0.1,

    attention_type="local",
    window_size=64, #! window size 10 -> 64
)

model = Seq2Seq(config).to(training_config["device"])
# model = model.to(torch.bfloat16)
# model.compile()
print(model)

train(
    model,
    train_set,
    valid_set,
    collate_fn,
    training_config,
    prefix="seq2seq_local_attention_64"
)

model.load_state_dict(torch.load(os.path.join("output", "seq2seq_local_attention_64", "best_model.pth")))
evaluate(
    model,
    valid_set,
    tokenizer,
    collate_fn,
    training_config
)

del model
torch.cuda.empty_cache()

Seq2Seq(
  (embedding): Embedding(65002, 512, padding_idx=65000)
  (encoder): Encoder(
    (encoder): LSTM(512, 512, num_layers=2, batch_first=True, dropout=0.1, bidirectional=True)
    (h_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
    (c_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
  )
  (decoder): Decoder(
    (decoder): LSTM(1024, 512, num_layers=2, batch_first=True, dropout=0.1)
    (attention): LocalAttention(
      (query_proj): Linear(in_features=512, out_features=512, bias=False)
      (key_proj): Linear(in_features=1024, out_features=512, bias=False)
      (value_proj): Linear(in_features=1024, out_features=512, bias=False)
      (output_proj): Linear(in_features=512, out_features=512, bias=False)
      (dropout): Dropout(p=0.1, inplace=False)
      (location_proj_up): Linear(in_features=512, out_features=512, bias=False)
      (location_proj_down): Linear(in_features=512, out_features=1, bias=False)
    )
    (dropout): Dropout(p=0.1,

Epoch 1/3:  33%|███▎      | 6250/18750 [1:18:02<2:38:18,  1.32it/s, val_loss: 2.7078]         

Validation Loss: 2.7078


Epoch 2/3:  33%|███▎      | 6250/18750 [1:18:03<2:38:18,  1.32it/s, val_loss: 2.7078]

Model Saved best loss: 2.7078


...

Model Saved best loss: 1.9850


Evaluating:   0%|          | 1/625 [00:06<1:04:12,  6.17s/it]

pred1:  ['5059 알루미늄 합금은 주로 마그네슘과 함께 이루어집니다. 열처리에 의해 강화되지 않고, 대신 수축 하딩이나 추운 기계적 작업으로 인해 더욱 강해집니다. 열 치료가 좋지 않아, 5059년 상반된 5059개의 합금이 획득되어 대부분의 기계적 강도를 유지하고 유지하는 데 기여했습니다. 5059개의 합금은 1999년 코루스 알룸니움에서 연구원들에 의해 독점적으로 5083 알루미늄 합금되었습니다.']
ans1:   ['5059 알루미늄 합금은 주로 마그네슘으로 합금된 알루미늄-마그네슘 합금입니다. 열처리로 강화되지 않고, 대신 재료의 변형 경화 또는 냉간 기계 가공으로 강해집니다. 열처리가 강도에 큰 영향을 미치지 않기 때문에 5059는 용접이 용이하고 기계적 강도를 대부분 유지할 수 있습니다. 5059 합금은 1999년 코러스 알루미늄의 연구원들에 의해 밀접하게 관련된 5083 알루미늄 합금에서 유래했습니다.']
pred2:  ['손베리 동물 보호 구역은 영국 사우스 요크셔주 로더햄에 위치한 중형 동물 구조 및 복지 자선 단체입니다. 동물 보호 및 쉼터이며, 빈민와 농장 동물을 위한 임시 쉼터와 영구적인 돌봄을 제공하며, 비파괴 정책에 입각한 쉼터와 농장 동물을 위한 임시 쉼터와 영구적인 돌봄을 제공합니다. 쏜베리는 1988년 스티브 밤포드에 의해 설립되었으며, 그의 어린 시절을 짓기 위해 동물을 돌보고 동물을 돌보고 돌보는 것을 촉구했습니다. 배포드는 5년 동안 소규모 카라반에 살았으며, 난방이 악화되었습니다. 노스 안스턴의 주요 보호구역 부지는 처음에는 케넬, 난방, 마구간, 헛간을 포함하여 5년 동안 살았습니다. 마구간은 나중에 워크솝의 버크스 농장으로 이전되었지만, 이 시설은 다시 매각되었고, 마구간은 다시 레이븐필드의 실버소프 농장에 있습니다.']
ans2:   ['쏜베리 동물 보호소는 영국 사우스 요크셔주 로더햄에 위치한 중간 규모의 동물 구조 및 복지 자선 단체입니다. 이곳은 동물 보호소이자 쉼터로, 애완 동물과 농장 동물에게 임시 

Evaluating: 100%|██████████| 625/625 [03:53<00:00,  2.67it/s]


BLEU: 0.1798
